# La misma imagen, otra biblioteca

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Nomenclatura oficial:** Procesamiento de Imágenes
* **Nombre de trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

Abrir una imagen con OpenCV, comprobar que es el mismo arreglo de NumPy que ya conocen de scikit-image y resolver la primera diferencia real entre las dos bibliotecas: el orden de los canales.

## Al terminar este cuaderno van a poder:

1. Cargar una imagen con `cv2.imread` y verificar que la lectura fue exitosa.
2. Describir una imagen por su forma, su tipo de dato y su rango de valores.
3. Explicar por qué una imagen leída con OpenCV se ve con los colores cambiados y corregirla con `cv2.cvtColor`.
4. Elegir la bandera de lectura adecuada para cada situación: color, grises o archivo sin cambios.
5. Escribir una función propia que cargue cualquier imagen lista para mostrar.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Biblioteca** | Conjunto de funciones ya escritas que se importa para no programar todo desde cero. |
| **Arreglo (`ndarray`)** | La tabla de números de NumPy donde vive una imagen: filas, columnas y canales. |
| **BGR** | El orden en que OpenCV guarda los canales: azul, verde, rojo. Es el orden inverso a RGB. |
| **Bandera de lectura** | Un valor que le indica a `cv2.imread` cómo abrir el archivo: en color, en grises o tal cual está. |
| **`uint8`** | Tipo de dato de números enteros entre 0 y 255. Es el formato más común para guardar píxeles. |
| **Metadatos EXIF** | Información que la cámara guarda dentro del archivo: fecha, modelo, orientación del celular. |

## Cambiar de biblioteca es traducir

**Analogía.** Una planilla de cálculo se puede abrir con Excel o con LibreOffice. Los datos son los mismos; lo que cambia es dónde está cada menú y cómo se llama cada opción. Pasar de scikit-image a OpenCV es algo parecido: la imagen sigue siendo una tabla de números, y lo que cambia son los nombres de las funciones y algunas convenciones.

**Bajada tecnológica.** OpenCV es la biblioteca de visión por computadora más usada en la industria: está detrás de cámaras de seguridad, robots de depósito, aplicaciones de escaneo de documentos y sistemas de asistencia al conductor. En esta materia la vamos a usar en tres niveles: esta introducción, las técnicas fundamentales de procesamiento y, más adelante, detección y video.

### ⚙ Configuración del entorno

Importamos las bibliotecas y verificamos sus versiones. Si alguna vez un resultado no coincide con el de este cuaderno, lo primero que conviene revisar es si la versión instalada es la misma.

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import skimage
from skimage import io

from utils import mostrar_imágenes

print(f"OpenCV {cv2.__version__} — NumPy {np.__version__} — scikit-image {skimage.__version__}")

Las tres bibliotecas quedaron importadas. `mostrar_imágenes` viene del archivo `utils.py` de esta carpeta: solo arma figuras de Matplotlib, para que el cuaderno se concentre en las imágenes y no en la configuración de los gráficos.

Ahora ubicamos la carpeta de imágenes del curso, que está un nivel más arriba que este cuaderno, y comprobamos que exista antes de seguir.

In [ ]:
CARPETA_IMÁGENES = Path("..") / "imagenes"
ruta_messi = CARPETA_IMÁGENES / "messi.jpg"

# 'exists' devuelve True si el archivo está en esa ruta.
assert ruta_messi.exists(), "No se encontró messi.jpg: revisen que la carpeta 'imagenes' esté junto a '001_intro'."

print(f"Imagen encontrada: {ruta_messi}")

El `assert` no mostró ningún mensaje de error, así que la ruta es correcta. Si el archivo faltara, el cuaderno se detendría acá con una explicación, en lugar de fallar más adelante con un error difícil de rastrear.

## 1. Leer una imagen con OpenCV

**Analogía.** Abrir un archivo de imagen es como desembalar un paquete: antes de usar lo que viene adentro, conviene revisar que el paquete haya llegado y que el contenido sea el esperado.

**Bajada tecnológica.** Todo sistema que procesa imágenes empieza igual: leer el archivo, verificar que se cargó y describir qué llegó. Un programa que no verifica este paso puede procesar durante horas una imagen vacía sin que nadie lo note.

### ✎ Antes de ejecutar

`cv2.imread` devuelve la imagen cargada. ¿Qué tipo de objeto esperan que sea? ¿Una "imagen" especial de OpenCV o algo que ya conocen?

In [ ]:
# 'imread' lee el archivo; 'IMREAD_COLOR' pide la imagen en color, con tres canales.
imagen_bgr = cv2.imread(filename=str(ruta_messi), flags=cv2.IMREAD_COLOR)

# Si el archivo no se pudo leer, OpenCV no da error: devuelve None.
assert imagen_bgr is not None, "OpenCV no pudo leer el archivo: revisen la ruta y el formato."

tipo_de_objeto = type(imagen_bgr)
print(f"Tipo de objeto: {tipo_de_objeto}")

El resultado es un `numpy.ndarray`: el mismo tipo de arreglo con el que trabajaron en scikit-image. OpenCV no inventa un formato propio, y por eso todo lo que ya saben de NumPy (indexar, recortar, calcular promedios) sigue funcionando.

Un detalle importante: cuando `cv2.imread` no encuentra el archivo, **no detiene el programa**, sino que devuelve `None`. Por eso el `assert` es obligatorio después de cada lectura.

Ahora describimos la imagen con las cuatro preguntas de siempre: qué forma tiene, qué tipo de dato usa y cuáles son su valor mínimo y máximo.

In [ ]:
forma = imagen_bgr.shape
tipo_de_dato = imagen_bgr.dtype
valor_mínimo = imagen_bgr.min()
valor_máximo = imagen_bgr.max()

print(f"Forma (alto, ancho, canales): {forma}")
print(f"Tipo de dato: {tipo_de_dato}")
print(f"Rango de valores: {valor_mínimo} a {valor_máximo}")

La imagen tiene 417 filas, 625 columnas y 3 canales, con valores enteros `uint8` entre 0 y 255. Es exactamente la misma descripción que obtendríamos con scikit-image. La diferencia no está en la forma del arreglo, sino en **qué significa cada canal**.

### ✎ Para pensar

- Si la imagen tiene 417 filas y 625 columnas, ¿la foto es horizontal o vertical?
- ¿Cuántos números en total guarda este arreglo? Hagan la cuenta con la forma antes de calcularla en código.

## 2. El orden de los canales: BGR

**Analogía.** Una fecha escrita como `03/04` significa 3 de abril en Argentina y 4 de marzo en Estados Unidos. Los números son los mismos; lo que cambia es la convención de lectura. Si alguien lee con la convención equivocada, la información llega completa, pero mal interpretada.

**Bajada tecnológica.** OpenCV guarda los canales en orden **azul, verde, rojo** por razones históricas de los primeros sistemas de captura con los que se desarrolló. Matplotlib y scikit-image, en cambio, usan rojo, verde, azul. Es la confusión más frecuente al empezar con OpenCV, y aparece también en sistemas en producción cuando se mezclan bibliotecas.

Para comprobarlo, miramos un píxel de la camiseta del jugador, que es azul oscura. Leemos sus tres valores en el orden en que los guarda OpenCV.

In [ ]:
fila_de_la_camiseta = 330
columna_de_la_camiseta = 330
píxel_de_la_camiseta = imagen_bgr[fila_de_la_camiseta, columna_de_la_camiseta]

# En OpenCV la posición 0 es azul, la 1 es verde y la 2 es rojo.
valor_azul = píxel_de_la_camiseta[0]
valor_verde = píxel_de_la_camiseta[1]
valor_rojo = píxel_de_la_camiseta[2]

print(f"Azul: {valor_azul} | Verde: {valor_verde} | Rojo: {valor_rojo}")

El valor más alto está en la **posición 0**, y la camiseta es azul: eso confirma que en OpenCV la posición 0 corresponde al azul. Si leyéramos ese mismo arreglo pensando que la posición 0 es rojo, concluiríamos que la camiseta es rojiza.

Eso es justamente lo que hace Matplotlib cuando recibe el arreglo sin convertir. Para verlo, preparamos una versión corregida con `cv2.cvtColor`, que reordena los canales de BGR a RGB.

In [ ]:
# 'cvtColor' convierte entre espacios de color; 'COLOR_BGR2RGB' invierte el orden de los canales.
imagen_rgb = cv2.cvtColor(src=imagen_bgr, code=cv2.COLOR_BGR2RGB)

paneles = [
    {"imagen": imagen_bgr, "título": "Sin convertir: BGR leído como RGB"},
    {"imagen": imagen_rgb, "título": "Convertida con COLOR_BGR2RGB"},
]
mostrar_imágenes(paneles)

En el panel izquierdo la camiseta azulgrana aparece con el rojo y el azul intercambiados, y la piel toma un tono azulado. El archivo no está roto: los números son correctos, pero Matplotlib los leyó con la convención equivocada. En el panel derecho, después de `cvtColor`, los colores coinciden con la escena real.

### ✎ Para pensar

- ¿Qué colores de la foto cambiaron menos al pasar de un panel al otro? ¿Por qué un color con rojo y azul parecidos casi no se altera?
- Si guardan `imagen_rgb` con una función de OpenCV que espera BGR, ¿qué creen que va a pasar con los colores del archivo guardado?

### ✎ Modificación guiada

La celda siguiente repite la lectura de un píxel. Cambien `fila_elegida` y `columna_elegida` por **215** y **490**, que caen sobre el brazalete de capitán, y antes de ejecutar anoten qué canal esperan que tenga el valor más alto.

In [ ]:
fila_elegida = 330
columna_elegida = 330
píxel_elegido = imagen_bgr[fila_elegida, columna_elegida]

valor_azul = píxel_elegido[0]
valor_verde = píxel_elegido[1]
valor_rojo = píxel_elegido[2]

print(f"Azul: {valor_azul} | Verde: {valor_verde} | Rojo: {valor_rojo}")

Con los valores originales la celda vuelve a mostrar el píxel de la camiseta. Cuando la modifiquen, comparen el resultado con su predicción: el brazalete es verde flúor, así que el canal dominante debería estar en la posición 1.

## 3. Traducción: esto ya lo vimos

En los cuadernos de fundamentos cargaron imágenes con `io.imread` de scikit-image. Leamos el mismo archivo con esa función para comparar las dos lecturas en números, no solo a simple vista.

In [ ]:
# 'io.imread' de scikit-image devuelve los canales en orden RGB.
imagen_skimage = io.imread(fname=str(ruta_messi))

forma_skimage = imagen_skimage.shape
tipo_skimage = imagen_skimage.dtype

print(f"scikit-image → forma: {forma_skimage}, tipo: {tipo_skimage}")
print(f"OpenCV       → forma: {forma}, tipo: {tipo_de_dato}")

Las dos bibliotecas devuelven un arreglo de la misma forma y el mismo tipo de dato. Falta comprobar si el contenido coincide: para eso comparamos el canal rojo de scikit-image contra la posición 2 de OpenCV, que según lo que vimos debería ser el rojo.

In [ ]:
rojo_skimage = imagen_skimage[:, :, 0].astype(dtype=int)
rojo_opencv = imagen_bgr[:, :, 2].astype(dtype=int)
azul_opencv = imagen_bgr[:, :, 0].astype(dtype=int)

# Promediamos la diferencia absoluta píxel a píxel: cero significa canales idénticos.
diferencia_con_rojo = abs(rojo_skimage - rojo_opencv).mean()
diferencia_con_azul = abs(rojo_skimage - azul_opencv).mean()

print(f"Rojo de scikit-image vs. posición 2 de OpenCV: {diferencia_con_rojo:.2f}")
print(f"Rojo de scikit-image vs. posición 0 de OpenCV: {diferencia_con_azul:.2f}")

La diferencia contra la posición 2 es prácticamente nula, mientras que contra la posición 0 es de decenas de niveles. Queda confirmado en números: **las dos bibliotecas leen los mismos píxeles y solo difieren en el orden de los canales**. Si la primera diferencia no diera exactamente cero, se debe a pequeñas variaciones entre los decodificadores de JPEG de cada biblioteca, no a un error de lectura.

Con esta comprobación podemos armar la tabla de traducción entre las dos bibliotecas:

| Tarea | scikit-image | OpenCV |
|---|---|---|
| Leer un archivo | `io.imread(fname=ruta)` | `cv2.imread(filename=ruta, flags=cv2.IMREAD_COLOR)` |
| Orden de los canales | RGB | BGR |
| Archivo inexistente | Detiene el programa con un error | Devuelve `None` sin avisar |
| Tipo del resultado | `numpy.ndarray` | `numpy.ndarray` |
| Reordenar canales | No hace falta | `cv2.cvtColor(src=imagen, code=cv2.COLOR_BGR2RGB)` |

## 4. Banderas de lectura

**Analogía.** En una fotocopiadora se puede pedir una copia en color o en blanco y negro. El original es el mismo; lo que cambia es qué copia nos llevamos. Las banderas de `cv2.imread` funcionan igual: deciden qué versión del archivo entra al programa.

**Bajada tecnológica.** Un sistema de lectura de patentes o de documentos suele leer directamente en grises, porque el color no aporta y ocupa tres veces más memoria. Un editor de imágenes, en cambio, necesita leer el archivo tal cual, con transparencia incluida.

### ✎ Antes de ejecutar

Vamos a leer el mismo archivo con `IMREAD_GRAYSCALE`. ¿Qué forma esperan que tenga el resultado? ¿Cambiará la cantidad de filas y columnas?

In [ ]:
# 'IMREAD_GRAYSCALE' convierte a grises durante la lectura: un solo canal.
imagen_gris = cv2.imread(filename=str(ruta_messi), flags=cv2.IMREAD_GRAYSCALE)
assert imagen_gris is not None, "No se pudo leer la imagen en grises."

forma_gris = imagen_gris.shape
print(f"Forma en color: {forma}")
print(f"Forma en grises: {forma_gris}")

Las filas y columnas se mantienen, pero desaparece la tercera dimensión: cada píxel pasa a tener un único valor de intensidad. La imagen en grises ocupa un tercio de la memoria de la versión en color.

Hay una tercera bandera que conviene conocer: `IMREAD_UNCHANGED`, que lee el archivo exactamente como está guardado. Además, OpenCV hace algo en silencio al leer en color: si el archivo trae metadatos EXIF de orientación, **gira la imagen automáticamente**. Las fotos de celular suelen traer esa marca.

Lo comprobamos leyendo con la bandera que desactiva ese giro, `IMREAD_IGNORE_ORIENTATION`, y comparando las formas.

In [ ]:
# El símbolo '|' combina dos banderas: color y sin aplicar la orientación EXIF.
banderas_sin_giro = cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION
imagen_sin_giro = cv2.imread(filename=str(ruta_messi), flags=banderas_sin_giro)

forma_sin_giro = imagen_sin_giro.shape
print(f"Con orientación EXIF aplicada: {forma}")
print(f"Ignorando la orientación EXIF: {forma_sin_giro}")

Las dos formas coinciden porque `messi.jpg` no trae orientación EXIF. Con una foto vertical tomada con un celular, en cambio, podrían no coincidir: la lectura por defecto la gira según lo que indica el celular, y la lectura que ignora la orientación la deja como fue guardada, a veces acostada.

| Bandera | Canales | Aplica orientación EXIF | Uso típico |
|---|---|---|---|
| `IMREAD_COLOR` | 3 (BGR) | Sí | Fotografías en general |
| `IMREAD_GRAYSCALE` | 1 | Sí | Documentos, bordes, detección |
| `IMREAD_UNCHANGED` | Los del archivo (incluye transparencia) | No | Íconos, PNG con fondo transparente |

### ✎ Para pensar

- Si un pipeline mide el ancho y el alto de fotos tomadas con celulares distintos, ¿por qué es importante saber si la orientación EXIF se aplicó antes de medir?
- ¿En qué situación preferirían leer en grises desde el principio en lugar de convertir después?

## 5. Del paso a paso a una función

Hasta acá repetimos siempre la misma secuencia: leer, verificar y convertir a RGB. Ahora que el procedimiento se entiende, lo encapsulamos en una función que vamos a reutilizar en los próximos cuadernos.

In [ ]:
def leer_imagen_rgb(ruta_de_la_imagen):
    """Lee una imagen con OpenCV, verifica la lectura y la devuelve en RGB.

    Recibe la ruta del archivo. Devuelve un arreglo (alto, ancho, 3) listo para mostrar.
    """
    imagen_leída = cv2.imread(filename=str(ruta_de_la_imagen), flags=cv2.IMREAD_COLOR)
    assert imagen_leída is not None, f"No se pudo leer: {ruta_de_la_imagen}"
    imagen_convertida = cv2.cvtColor(src=imagen_leída, code=cv2.COLOR_BGR2RGB)
    return imagen_convertida

print("Función leer_imagen_rgb definida.")

La función recibe una ruta y devuelve la imagen en RGB, con la verificación incluida. Probémosla con dos imágenes nuevas de la carpeta del curso.

In [ ]:
imagen_globos = leer_imagen_rgb(CARPETA_IMÁGENES / "globos.jpg")
imagen_cartas = leer_imagen_rgb(CARPETA_IMÁGENES / "cards.jpg")

paneles = [
    {"imagen": imagen_globos, "título": "globos.jpg"},
    {"imagen": imagen_cartas, "título": "cards.jpg"},
]
mostrar_imágenes(paneles)

Las dos imágenes se ven con sus colores reales, sin que tengamos que repetir la conversión: el cielo de los globos es celeste y los palos rojos de las cartas se ven rojos. La función hace el trabajo de las tres líneas anteriores con una sola llamada.

### ✎ Completado

Escriban una segunda función, `describir_imagen`, que reciba una imagen y muestre las cuatro descripciones que usamos en la sección 1. Descomenten las líneas y completen los `...`.

In [ ]:
# ─── Completen los pasos faltantes ───────────────────────────────────────────
#
# TODO 1: guarden en 'forma' la forma de la imagen recibida.
# TODO 2: guarden en 'tipo_de_dato', 'valor_mínimo' y 'valor_máximo' lo que corresponde.
# TODO 3: muestren los cuatro valores con print, como en la sección 1.
#
# def describir_imagen(imagen):
#     """Muestra la forma, el tipo de dato y el rango de valores de una imagen."""
#     forma = ...
#     tipo_de_dato = ...
#     ...
#
# describir_imagen(imagen_cartas)
# ─────────────────────────────────────────────────────────────────────────────

Cuando la función funcione, pruébenla con `imagen_globos` y con `imagen_gris`. ¿Qué cambia en la descripción de la imagen en grises?

## Actividad de producción

In [ ]:
# ─── Espacio de práctica ─────────────────────────────────────────────────────
#
# Consigna: comprueben la traducción entre bibliotecas con otra imagen.
#   1. Elijan una imagen de la carpeta 'imagenes' distinta de messi.jpg.
#   2. Léanla con io.imread y con leer_imagen_rgb.
#   3. Calculen la diferencia media entre los canales rojos de las dos lecturas.
#   4. Escriban en una celda de texto si la traducción se sostiene y qué
#      diferencia encontraron con respecto a messi.jpg.
#
# ─────────────────────────────────────────────────────────────────────────────

## Cierre

OpenCV no cambió lo que es una imagen: sigue siendo un arreglo de NumPy con filas, columnas y canales. Lo que cambiaron fueron las convenciones, y la más importante es el orden BGR.

| Idea | Qué recordar |
|---|---|
| Tipo de dato | `cv2.imread` devuelve un `numpy.ndarray`, igual que scikit-image. |
| Lectura fallida | Devuelve `None` sin error: siempre verificar con `assert`. |
| Orden de canales | BGR en OpenCV, RGB en Matplotlib y scikit-image. |
| Conversión | `cv2.cvtColor(src=imagen, code=cv2.COLOR_BGR2RGB)` antes de mostrar. |
| Banderas | Color, grises o sin cambios; en color se aplica la orientación EXIF. |

**Limitaciones de lo que vimos.** La conversión a RGB solo corrige la visualización: si mezclan funciones de OpenCV con funciones de otras bibliotecas, tienen que saber en todo momento en qué orden está cada arreglo. Tampoco probamos todavía una imagen con transparencia, que necesita `IMREAD_UNCHANGED`.

En el próximo cuaderno, `002_canales`, vamos a separar la imagen en sus canales y a describir el color de otra manera: con el espacio HSV.